Hi there!
This is the code template for CW2 task3 of COMP34711 2025/26.

- <span style="color:red; font-size:1em">First of all, please rename the notebook into "{your_student_id}_CW2_task{your_task_number}.ipynb", for example "12345678_CW2_task3.ipynb".</span>

- In this template, we only provide the minimal structure for your coursework.
  
- Please carefully read and organize your code in the template we provided.

## Constants

In [1]:
#Please keep only necessary information in this cell.

#----------------------Please keep all following constants unchanged.----------------------------------------
NUM_ROWS_VALIDATION = 1031 # Number of rows in validation set
NUM_ROWS_TEST = 1053 # Number of rows in test set

#----------------------Please modify the following constants to fit your actual value.-----------------------
STUDENT_ID = '10879360'  # Replace with your actual 8-digits student ID
TRAINING_SET = './data/CW2_training_dataset.csv' # Replace with the actual path to your training dataset csv file
VALIDATION_SET = './data/CW2_validation_dataset.csv'  # Replace with the actual path to your validation dataset csv file
VALIDATION_SET_OUTPUT = f'./data/{STUDENT_ID}_CW2_task3_validation_results.csv'  # Replace with the actual path to your validation prediction csv file
TEST_SET_INPUT = './data/CW2_test_dataset.csv'  # Replace with the actual path to your test prediction csv file

#----------------------Your constants------------------------------------------------
# By adding more constants here, you can help improve the clarity and maintainability of your code and make the reviewing easier for TAs.

## Installations

In [2]:
# Install required packages for the coursework
# Uncomment and run the following lines if needed

!pip install pandas scikit-learn torch nltk --quiet
!pip install transformers accelerate datasets evaluate --quiet
!pip install sentencepiece protobuf --quiet

## Imports

In [3]:
#Please keep all imports of your code cells in this cell

#---------------------Required imports----------------------
import pandas as pd
import re
import sys
import os.path
import csv
from sklearn.metrics import f1_score
#----------------------Your imports-------------------------
import numpy as np

# Pytorch tools
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler

# Transformer tools
from transformers import DebertaV2Tokenizer, DebertaV2Model

# Fix seed for Ensemble
def set_seed(seed_value=99):
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Check a device that will run this code below
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## Start of your code cells

- The code cells provided below are demo code format for TAs to quickly locate your implementation.

- You have full right to freely add/delete/edit the titles and codes in the following cells.

- Please follow this genre order: "comedy, cult, flashback, historical, revenge, romantic, scifi, violence".

### Data Loading

In [4]:
# --------------------------------------------------------------------------------
# 1. Configuration & Tokenizer Initialization
# --------------------------------------------------------------------------------
MAX_LEN = 512
BATCH_SIZE = 12  # 12 for Tesla 4 GPU
MODEL_NAME = 'microsoft/deberta-v3-base'

print(f"Loading {MODEL_NAME} tokenizer~")
tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)

# --------------------------------------------------------------------------------
# 2. Raw Data Loading (Clean Version)
# --------------------------------------------------------------------------------
# Load CSV files
df_train = pd.read_csv(TRAINING_SET)
df_validation = pd.read_csv(VALIDATION_SET)

print(f"✅ Raw Data Loaded (Train: {len(df_train)}, Validation: {len(df_validation)})")
print(f"   - Tokenizer initialized: {MODEL_NAME}")

Loading microsoft/deberta-v3-base tokenizer~


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

✅ Raw Data Loaded (Train: 7127, Validation: 1031)
   - Tokenizer initialized: microsoft/deberta-v3-base


### Tokenization

In [5]:
# --------------------------------------------------------------------------------
# 1. Preprocessing Function Definition (Task 3 Optimized: Head+Tail + Separator)
# --------------------------------------------------------------------------------
def preprocess_data(df, tokenizer, max_len):
    """
    [Task 3 Optimized] Preprocessing for DeBERTa-v3.
    - Applies Head+Tail Truncation to preserve the most important parts of long texts.
    - Uses the [SEP] token to explicitly separate the Movie Title from the Plot Synopsis.
    - Automatically retrieves special tokens ([CLS], [SEP], [PAD]) from the tokenizer.
    """
    print(f"Tokenizing {len(df)} samples (Strategy: Head + Tail + Separator)~")

    input_ids_list = []
    attention_masks_list = []

    # Retrieve DeBERTa Special Token IDs directly from the tokenizer
    CLS_TOKEN = tokenizer.cls_token_id # [CLS]
    SEP_TOKEN = tokenizer.sep_token_id # [SEP]
    PAD_TOKEN = tokenizer.pad_token_id # [PAD]

    # [Config] Head and Tail allocation
    # Total available slots = 512 - 2 (for CLS and SEP) = 510
    # Allocation: 128 for Head (Title/Intro) + 382 for Tail (Resolution)
    HEAD_LEN = 128
    TAIL_LEN = max_len - 2 - HEAD_LEN # 512 - 2 - 128 = 382

    titles = df['title'].astype(str).tolist()
    plots = df['plot_synopsis'].astype(str).tolist()

    for title, plot in zip(titles, plots):
        # 1. Tokenize title and plot separately
        # Use add_special_tokens=False to get raw IDs without CLS/SEP
        title_tokens = tokenizer.encode(title, add_special_tokens=False)
        plot_tokens = tokenizer.encode(plot, add_special_tokens=False)

        # 2. [Key Step 1] Insert Separator (Title + [SEP] + Plot)
        # This helps the model distinguish between the title and the plot context.
        combined_tokens = title_tokens + [SEP_TOKEN] + plot_tokens

        # 3. [Key Step 2] Apply Head + Tail Truncation
        if len(combined_tokens) > max_len - 2:
            # If sequence exceeds limit, keep the beginning (Head) and end (Tail), discarding the middle.
            combined_tokens = combined_tokens[:HEAD_LEN] + combined_tokens[-TAIL_LEN:]

        # 4. Add Special Tokens and Padding
        # Final Structure: [CLS] + (Title + [SEP] + Plot_Head + Plot_Tail) + [SEP]
        final_tokens = [CLS_TOKEN] + combined_tokens + [SEP_TOKEN]

        pad_len = max_len - len(final_tokens)
        input_ids = final_tokens + [PAD_TOKEN] * pad_len
        attention_mask = [1] * len(final_tokens) + [0] * pad_len

        input_ids_list.append(input_ids)
        attention_masks_list.append(attention_mask)

    # Convert lists to tensors
    input_ids = torch.tensor(input_ids_list, dtype=torch.long)
    attention_masks = torch.tensor(attention_masks_list, dtype=torch.long)

    # Process Labels if available
    if len(df.columns) > 3:
        labels = torch.tensor(df.iloc[:, 3:].values, dtype=torch.float)
    else:
        labels = None

    return input_ids, attention_masks, labels

# --------------------------------------------------------------------------------
# 2. Preprocessing Execution & DataLoader Creation
# --------------------------------------------------------------------------------
# Execute Preprocessing using the DeBERTa tokenizer defined earlier
train_input_ids, train_attention_masks, train_labels = preprocess_data(df_train, tokenizer, MAX_LEN)
val_input_ids, val_attention_masks, val_labels = preprocess_data(df_validation, tokenizer, MAX_LEN)

print("✅ Preprocessing complete. Tensors created.")

# Create TensorDatasets
train_dataset = TensorDataset(train_input_ids, train_attention_masks, train_labels)
val_dataset = TensorDataset(val_input_ids, val_attention_masks, val_labels)

# Create DataLoaders (num_workers=2 is recommended for Colab)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✅ DataLoaders Ready (Max Length: {MAX_LEN}, Batch Size: {BATCH_SIZE})")

Tokenizing 7127 samples (Strategy: Head + Tail + Separator)~
Tokenizing 1031 samples (Strategy: Head + Tail + Separator)~
✅ Preprocessing complete. Tensors created.
✅ DataLoaders Ready (Max Length: 512, Batch Size: 12)


### Model and Training

In [6]:
# --------------------------------------------------------------------------------
# 1. Asymmetric Loss Class (ASL)
# --------------------------------------------------------------------------------
class AsymmetricLoss(nn.Module):
    # Asymmetric Loss to handle class imbalance by down-weighting easy negatives.
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super(AsymmetricLoss, self).__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, x, y):
        targets = y
        xs_pos = torch.sigmoid(x)
        xs_neg = 1.0 - xs_pos

        # Margin clipping for negatives
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)

        # Standard Cross-Entropy parts
        los_pos = targets * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1.0 - targets) * torch.log(xs_neg.clamp(min=self.eps))

        # Asymmetric weights
        pos_weight = targets * (1.0 - xs_pos).pow(self.gamma_pos)
        neg_weight = (1.0 - targets) * xs_pos.pow(self.gamma_neg)

        loss = pos_weight * los_pos + neg_weight * los_neg
        return -loss.mean()

# --------------------------------------------------------------------------------
# 2. Model Definition: DeBERTa V3 with Mean Pooling
# --------------------------------------------------------------------------------
class DebertaMeanPoolingClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout_rate=0.1):
        super(DebertaMeanPoolingClassifier, self).__init__()
        # [Change] Load Base Model (DebertaV2Model) instead of SequenceClassification
        # This allows access to all hidden states for Mean Pooling.
        self.deberta = DebertaV2Model.from_pretrained(
            model_name,
            output_attentions=False,
            output_hidden_states=False
        )

        # Classifier Head
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.deberta.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        # 1. Forward pass through Base Model
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state # [Batch, Len, Hidden]

        # 2. Mean Pooling Logic
        # Expand Attention Mask to match Hidden State dimensions
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()

        # (1) Sum embeddings of valid tokens (ignoring padding)
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)

        # (2) Count valid tokens
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9) # Prevent division by zero

        # (3) Calculate Mean
        mean_embeddings = sum_embeddings / sum_mask

        # 3. Classifier
        logits = self.classifier(self.dropout(mean_embeddings))

        return logits

# --------------------------------------------------------------------------------
# 3. Training & Evaluation Functions
# --------------------------------------------------------------------------------
def train_epoch(model, iterator, optimizer, scheduler, criterion, device, scaler):
    # Train the model for one epoch using Mixed Precision.
    model.train()
    epoch_loss = 0

    for batch in iterator:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()

        # Mixed Precision Forward Pass
        with autocast(device_type='cuda'):
            predictions = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(predictions, labels)

        # Backward Pass with Scaler
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion, device):
    # Evaluate model and find optimal thresholds per class.
    model.eval()
    epoch_loss = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in iterator:
            input_ids = batch[0].to(device)
            attention_mask = batch[1].to(device)
            labels = batch[2].to(device)

            with autocast(device_type='cuda'):
                predictions = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(predictions, labels)

            epoch_loss += loss.item()
            probs = torch.sigmoid(predictions)
            all_probs.extend(probs.cpu().float().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    # Search for Optimal Thresholds (Class-wise)
    best_thresholds = []
    for i in range(8):
        best_f1 = 0.0
        best_t = 0.5
        # Search range: 0.30 to 0.70
        for t in np.arange(0.3, 0.71, 0.01):
            y_pred = (all_probs[:, i] > t).astype(int)
            f1 = f1_score(all_labels[:, i], y_pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
        best_thresholds.append(best_t)

    # Calculate Weighted F1 with optimal thresholds
    final_preds = np.zeros_like(all_probs)
    for i in range(8):
        final_preds[:, i] = (all_probs[:, i] > best_thresholds[i]).astype(int)

    final_f1_weighted = f1_score(all_labels, final_preds, average='weighted', zero_division=0)

    return epoch_loss / len(iterator), final_f1_weighted, best_thresholds

# --------------------------------------------------------------------------------
# LLRD Optimizer Group (Layer-wise Learning Rate Decay)
# --------------------------------------------------------------------------------
def get_optimizer_grouped_parameters(model, learning_rate, weight_decay, layer_decay=0.9):
    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = []

    # [Modified] Access the base model directly from self.deberta
    base_model = model.deberta

    # 1. Embeddings
    optimizer_grouped_parameters.append({
        "params": [p for n, p in base_model.embeddings.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": weight_decay,
        "lr": learning_rate * (layer_decay ** 13)
    })

    # 2. Encoder Layers (0~11)
    for layer_idx, layer in enumerate(base_model.encoder.layer):
        decay_power = 12 - layer_idx
        lr_layer = learning_rate * (layer_decay ** decay_power)

        optimizer_grouped_parameters.append({
            "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay, "lr": lr_layer
        })
        optimizer_grouped_parameters.append({
            "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0, "lr": lr_layer
        })

    # 3. Classifier Head (Parameters not in base_model)
    head_params = [p for n, p in model.named_parameters() if "deberta" not in n]
    optimizer_grouped_parameters.append({"params": head_params, "weight_decay": 0.0, "lr": learning_rate})

    return optimizer_grouped_parameters

# --------------------------------------------------------------------------------
# 4. Main Execution Block (Ensemble Training)
# --------------------------------------------------------------------------------

# 1. Hyperparameters
MODEL_NAME = 'microsoft/deberta-v3-base'
NUM_LABELS = 8
DROPOUT_RATE = 0.1
LEARNING_RATE = 1e-5
N_EPOCHS = 10

# Seeds for Ensemble
SEEDS = [16, 42, 378]

# Global variables to store results
ensemble_best_thresholds = None
global_best_f1 = 0.0

print(f"🚀 Starting Ensemble Training for Seeds: {SEEDS}~")
print(f"   Model: DeBERTa V3 + Mean Pooling + LLRD + ASL")

for seed in SEEDS:
    print(f"\n" + "="*80)
    print(f"   🌱 Training Seed: {seed}")
    print("="*80)

    # 1. Set Seed for Reproducibility
    set_seed(seed)

    # 2. Initialize Model (Mean Pooling Class)
    model = DebertaMeanPoolingClassifier(MODEL_NAME, NUM_LABELS, DROPOUT_RATE)
    model = model.to(device)

    # Initialize LLRD Optimizer
    grouped_params = get_optimizer_grouped_parameters(model, LEARNING_RATE, 0.01, layer_decay=0.95)
    optimizer = torch.optim.AdamW(grouped_params)

    # Scheduler & Loss
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05).to(device)
    scaler = GradScaler('cuda')

    # 3. Training Loop for current seed
    seed_best_f1 = 0.0

    print(f"{'Epoch':^6} | {'Train Loss':^10} | {'Val Loss':^10} | {'Val F1 (W)':^12} | {'Result':^10}")
    print("-" * 80)

    for epoch in range(N_EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, device, scaler)
        val_loss, val_f1, current_threshs = evaluate(model, val_loader, criterion, device)

        scheduler.step(val_f1)

        # Save Best Model for this Seed
        if val_f1 > seed_best_f1:
            seed_best_f1 = val_f1
            # Save file with seed in filename
            save_path = f'{STUDENT_ID}_task3_best_model_seed{seed}.pt'
            torch.save(model.state_dict(), save_path)
            saved_msg = "🔥 Saved"

            # Update Global Best Thresholds (for ensemble usage)
            if seed_best_f1 > global_best_f1:
                global_best_f1 = seed_best_f1
                ensemble_best_thresholds = current_threshs
        else:
            saved_msg = ""

        print(f"{epoch+1:02}/{N_EPOCHS:02}  | {train_loss:.4f}     | {val_loss:.4f}     | {val_f1:.4f}       | {saved_msg}")

    print(f"\n✅ Seed {seed} Best F1: {seed_best_f1:.4f} -> Saved to {save_path}")

print("\n🎉 All 3 Ensemble Models Trained Successfully!")
print(f"✅ Global Best Single F1: {global_best_f1:.4f}")
print(f"✅ Recommended Thresholds for Ensemble: {ensemble_best_thresholds}")

🚀 Starting Ensemble Training for Seeds: [16, 42, 378]~
   Model: DeBERTa V3 + Mean Pooling + LLRD + ASL

   🌱 Training Seed: 16


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Epoch  | Train Loss |  Val Loss  |  Val F1 (W)  |   Result  
--------------------------------------------------------------------------------


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

01/10  | 0.0980     | 0.0839     | 0.5501       | 🔥 Saved
02/10  | 0.0828     | 0.0799     | 0.5783       | 🔥 Saved
03/10  | 0.0790     | 0.0769     | 0.5971       | 🔥 Saved
04/10  | 0.0762     | 0.0764     | 0.6077       | 🔥 Saved
05/10  | 0.0738     | 0.0750     | 0.6110       | 🔥 Saved
06/10  | 0.0713     | 0.0759     | 0.6164       | 🔥 Saved
07/10  | 0.0693     | 0.0749     | 0.6131       | 
08/10  | 0.0675     | 0.0770     | 0.6217       | 🔥 Saved
09/10  | 0.0655     | 0.0752     | 0.6223       | 🔥 Saved
10/10  | 0.0631     | 0.0793     | 0.6173       | 

✅ Seed 16 Best F1: 0.6223 -> Saved to 10879360_task3_best_model_seed16.pt

   🌱 Training Seed: 42
Epoch  | Train Loss |  Val Loss  |  Val F1 (W)  |   Result  
--------------------------------------------------------------------------------
01/10  | 0.0934     | 0.0852     | 0.5322       | 🔥 Saved
02/10  | 0.0849     | 0.0816     | 0.5760       | 🔥 Saved
03/10  | 0.0810     | 0.0783     | 0.5906       | 🔥 Saved
04/10  | 0.0785    

### Prediction

In [11]:
# --------------------------------------------------------------------------------
# Task 3 Ensemble Prediction Setup
# --------------------------------------------------------------------------------
# 1. Define paths for the 3 ensemble models (Must match saved filenames)
ENSEMBLE_MODELS = [
    f'{STUDENT_ID}_task3_best_model_seed16.pt',
    f'{STUDENT_ID}_task3_best_model_seed42.pt',
    f'{STUDENT_ID}_task3_best_model_seed378.pt'
]

# 2. Optimal Thresholds (Use values from training output)
OPTIMAL_THRESHOLDS = ensemble_best_thresholds

# --------------------------------------------------------------------------------
# Ensemble Prediction Execution Function
# --------------------------------------------------------------------------------
def generate_ensemble_prediction(input_file, output_file, thresholds_list):
    print(f"\n--- [Task 3] Ensemble Prediction for {input_file} ---")
    df = pd.read_csv(input_file)

    # 1. Data Preprocessing (Same strategy as training: Head+Tail+Separator)
    print("Tokenizing data~")
    input_ids, attention_masks, _ = preprocess_data(df, tokenizer, MAX_LEN)
    dataset = TensorDataset(input_ids, attention_masks)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    # 2. Initialize array to accumulate probabilities
    # Shape: (Num Samples, 8 Genres)
    total_probs = np.zeros((len(df), 8), dtype=float)

    # 3. Inference Loop (Iterate through each model)
    for model_path in ENSEMBLE_MODELS:
        print(f"Running inference with: {model_path}")

        try:
            # Initialize architecture and load weights
            model = DebertaMeanPoolingClassifier(MODEL_NAME, NUM_LABELS, DROPOUT_RATE).to(device)
            model.load_state_dict(torch.load(model_path, map_location=device))
            model.eval()
        except FileNotFoundError:
            print(f"❌ Error: Model file not found: {model_path}")
            return

        temp_probs = []
        with torch.no_grad():
            for batch in loader:
                b_input_ids = batch[0].to(device)
                b_input_mask = batch[1].to(device)

                with autocast(device_type='cuda'):
                    outputs = model(b_input_ids, b_input_mask)

                # Calculate Probabilities (Sigmoid)
                probs = torch.sigmoid(outputs)
                # Convert FP16 to FP32 and store
                temp_probs.extend(probs.cpu().float().numpy())

        # Accumulate probabilities from current model
        total_probs += np.array(temp_probs)
        print(f"   -> Completed.")

    # 4. Soft Voting (Average probabilities)
    avg_probs = total_probs / len(ENSEMBLE_MODELS)

    # 5. Apply Class-wise Thresholds
    print(f"Applying Class-wise thresholds: {thresholds_list}")

    final_preds = np.zeros_like(avg_probs, dtype=int)
    for i in range(8):
        # Apply threshold for each genre (Prob > Threshold -> 1, else 0)
        final_preds[:, i] = (avg_probs[:, i] > thresholds_list[i]).astype(int)

    # 6. Save Results
    ids = df.iloc[:, 0].values
    genre_columns = ['comedy', 'cult', 'flashback', 'historical', 'revenge', 'romantic', 'scifi', 'violence']

    df_output = pd.DataFrame(final_preds, columns=genre_columns)
    df_output = df_output.astype(int) # Ensure integer type
    df_output.insert(0, 'ID', ids) # Add ID column

    # Save to CSV (No header, No index)
    df_output.to_csv(output_file, index=False, header=False)

    print(f"\n✅ Ensemble Prediction Saved to: {output_file}")
    print(f"Total Rows: {len(df_output)}")
    print(f"Check First Row: {df_output.iloc[0].values}")

    return df_output

# Execution
# 1. Test on Validation Set first
# df_output = generate_ensemble_prediction(VALIDATION_SET, VALIDATION_SET_OUTPUT, OPTIMAL_THRESHOLDS)

# 2. For Final Submission
df_output = generate_ensemble_prediction(TEST_SET_INPUT, f'./{STUDENT_ID}_CW2_task3_results_prediction.csv', OPTIMAL_THRESHOLDS)


--- [Task 3] Ensemble Prediction for ./data/CW2_test_dataset.csv ---
Tokenizing data~
Tokenizing 1053 samples (Strategy: Head + Tail + Separator)~
Running inference with: 10879360_task3_best_model_seed16.pt
   -> Completed.
Running inference with: 10879360_task3_best_model_seed42.pt
   -> Completed.
Running inference with: 10879360_task3_best_model_seed378.pt
   -> Completed.
Applying Class-wise thresholds: [np.float64(0.5300000000000002), np.float64(0.5200000000000002), np.float64(0.5600000000000003), np.float64(0.4200000000000001), np.float64(0.49000000000000016), np.float64(0.5900000000000003), np.float64(0.47000000000000014), np.float64(0.5700000000000003)]

✅ Ensemble Prediction Saved to: ./10879360_CW2_task3_results_prediction.csv
Total Rows: 1053
Check First Row: ['9484ac61-0e30-4799-9998-6f74f4cbb204' np.int64(0) np.int64(0)
 np.int64(0) np.int64(0) np.int64(1) np.int64(0) np.int64(0) np.int64(1)]


## End of your code cells

### Evaluation scripts

In [8]:
def read_data(submission_file_path, gold_standard_file_path):
    """
    Read submission and gold standard files.
    Extract student ID from filename.
    """
    # Try to find student ID from the filename (looks for 8 digit numbers)
    id_regex = r'\d{8}'

    user_id = re.findall(id_regex, submission_file_path)
    print("Found your ID: ", user_id)
    if user_id:
        user_id = user_id[0]
    else:
        user_id = 'Unknown'

    # Load submission CSV
    print(f"\nLoading submission file: {submission_file_path}")
    submission_df = pd.read_csv(submission_file_path, sep=',', header=None,
                                quoting=csv.QUOTE_NONE, encoding='utf-8')

    # Load gold standard CSV
    print(f"Loading gold standard file: {gold_standard_file_path}")
    gold_standard_df = pd.read_csv(gold_standard_file_path, header=None)

    # Remove columns 1 and 2 (keep only ID and labels)
    gold_standard_df = gold_standard_df.drop([1, 2], axis=1)
    # Skip header row
    gold_standard_df = gold_standard_df.iloc[1:]

    return submission_df, gold_standard_df, user_id


def match_and_prepare_data(submission_df, gold_standard_df, user_id):
    """
    Match submission rows with gold standard rows by ID.
    Prepare data for evaluation.
    """
    gold_standard_labels = []
    submission_labels = []
    missed_rows = []
    submission_df_copy = submission_df.copy()

    print(f"\nMatching submission with gold standard...")
    print(f"Gold standard rows: {len(gold_standard_df)}")
    print(f"Submission rows: {len(submission_df_copy)}")

    # Match each gold standard row with submission
    for index, row in gold_standard_df.iterrows():
        row = row.reset_index(drop=True)
        row_found = False
        row_id = row[0]

        # Extract gold standard labels
        row_labels = [int(row[i]) for i in range(1, len(row))]
        gold_standard_labels.append(row_labels)

        # Find corresponding submission row
        for sub_index, submission_row in submission_df_copy.iterrows():
            if submission_row[0].strip() == row_id.strip():
                try:
                    # Extract submission labels
                    submission_row_labels = [int(submission_row[i]) for i in range(1, len(submission_row))]
                except:
                    # Handle malformed labels (take first character if multi-digit)
                    submission_row_labels = [int(str(submission_row[i])[0]) for i in range(1, len(submission_row))]

                submission_labels.append(submission_row_labels)
                row_found = True
                submission_df_copy.drop(sub_index, inplace=True)
                break

        if not row_found:
            # If row is missing, add inverse labels (worst possible prediction)
            missed_rows.append(row_id)
            submission_labels.append([0 if label == 1 else 1 for label in row_labels])

    return gold_standard_labels, submission_labels, missed_rows


def evaluate_submission(gold_standard_labels, submission_labels):
    """
    Calculate weighted F1 score.
    """
    print(f"\nCalculating weighted F1 score...")

    # Calculate weighted F1 score (accounts for class imbalance)
    f1_weighted = f1_score(gold_standard_labels, submission_labels, average='weighted')

    return f1_weighted


def print_results(user_id, f1_weighted, missed_rows):
    """
    Print evaluation results to screen.
    """
    print("\n" + "="*70)
    print("YOUR SUBMISSION EVALUATION REPORT")
    print("="*70)

    # Alert if ID not found in filename
    if user_id == 'Unknown':
        print('WARNING: ID not found in filename!')
        print('   Please ensure your filename contains your 8-digit student ID.')
        print()

    print(f"Your ID: {user_id}")
    print()

    # Display F1 score with visual indicator
    print("EVALUATION RESULTS:")
    print(f"   Weighted F1 Score: {f1_weighted:.4f}")
    print()

    # Report missing rows
    if missed_rows:
        print(f"MISSING DATA ({len(missed_rows)} rows not found):")
        print("-" * 70)
        for i, row in enumerate(missed_rows[:10], 1):  # Show first 10
            print(f"    {i}. Row ID: {row}")
        if len(missed_rows) > 10:
            print(f"    ... and {len(missed_rows) - 10} more missing rows")
        print()
        print("TIP: Make sure your submission includes all required rows.")
        print("        Missing rows are penalized with worst possible predictions.")
    else:
        print("DATA COMPLETENESS: All expected rows found in your submission!")

    print()
    print("="*70)
    print()


def evaluate(submission_path, gold_standard_path):
    """
    Main function to run the submission evaluation script.
    """

    submission_file = submission_path
    gold_standard_file = gold_standard_path

    # Check if files exist
    if not os.path.exists(submission_file):
        print(f"Error: Your submission file '{submission_file}' not found!")
        print("Make sure the file path is correct and the file exists.")
        sys.exit(1)

    if not os.path.exists(gold_standard_file):
        print(f"Error: Gold standard file '{gold_standard_file}' not found!")
        print("Make sure you have the correct gold standard file.")
        sys.exit(1)

    try:
        # Step 1: Read data
        submission_df, gold_standard_df, user_id = read_data(submission_file, gold_standard_file)

        # Step 2: Match and prepare data
        gold_standard_labels, submission_labels, missed_rows = match_and_prepare_data(
            submission_df, gold_standard_df, user_id
        )

        # Step 3: Evaluate
        f1_weighted = evaluate_submission(gold_standard_labels, submission_labels)

        # Step 4: Print results
        print_results(user_id, f1_weighted, missed_rows)

    except Exception as e:
        print(f"Error during evaluation: {str(e)}")
        print("Please check that your files are in the correct CSV format.")
        print("Each row should contain: ID, label1, label2, label3, ...")
        import traceback
        traceback.print_exc()
        sys.exit(1)

### Evaluate the model on the validation dataset

In [9]:
# Please run the evaluation scripts cell above before running the mark_and_record

# Please make sure that output format is like following (no header row, no tilte and plot columns):
# 94834c61-0e30-4799-9998-6f74f6sbb204	0	1	0	0	1	0	0	0
# 559sdd28-b6a2-4662-ab55-a6678as26a56	0	0	0	0	0	0	1	0
# b71y3317-04cd-42f5-a380-d21dfasdbd36	0	0	0	0	1	0	0	0

evaluation_results = evaluate(VALIDATION_SET_OUTPUT, VALIDATION_SET)

Found your ID:  ['10879360']

Loading submission file: ./data/10879360_CW2_task3_validation_results.csv
Loading gold standard file: ./data/CW2_validation_dataset.csv

Matching submission with gold standard...
Gold standard rows: 1031
Submission rows: 1031

Calculating weighted F1 score...

YOUR SUBMISSION EVALUATION REPORT
Your ID: 10879360

EVALUATION RESULTS:
   Weighted F1 Score: 0.6142

DATA COMPLETENESS: All expected rows found in your submission!




### Save predictions to formatted file.

In [12]:
# Now please modify the code to format your output csv file.

# Please make sure that output format is like following (no header row, no title and plot columns):
# 94834c61-0e30-4799-9998-6f74f6sbb204	0	1	0	0	1	0	0	0
# 559sdd28-b6a2-4662-ab55-a6678as26a56	0	0	0	0	0	0	1	0
# b71y3317-04cd-42f5-a380-d21dfasdbd36	0	0	0	0	1	0	0	0

output_df = df_output  # Replace with your actual DataFrame or output
# For example, if you have a DataFrame named 'output_df', you can save it
assert isinstance(output_df, pd.DataFrame)
assert len(output_df) == NUM_ROWS_TEST, "Output length is not aligned with the testdata.csv."
assert len(output_df.columns) == 9, "Please make sure to follow the format above and keep only IDs and 8 columns of prediction."
output_df.to_csv(f'./{STUDENT_ID}_CW2_task3_results.csv', index=False, header=False)